### 理解CO3工作过程

为变量生成符号映射，并混合执行（Concolic Execution），边执行边求解。Symbolic Value Flow Graph（符号值流图），是 CO3 编译期从固件源码里提取的「符号输入流向模板图」

SVFG 是有向无环图（SSA 形式的数据流切片）

``` C
addr = sym_read(0);       // 叶子：MEM_R（未知）
len = sym_read(1);         // 叶子：MEM_R（未知）
addr_ok = (addr == 0x5A); // 计算：EQ（常量0x5A已知）
len_ok = (len > 0 && len <=4); // 计算：GT+AND（常量0/4已知）
if (addr_ok && len_ok) {  // 根：BRANCH（漏洞分支）
    trigger_bug();
}
```
提取的符号：
``` python
MEM_R(addr) → EQ(0x5A) → BRANCH(addr_ok)
MEM_R(len) → GT(0) → AND(<=4) → BRANCH(len_ok)
BRANCH(addr_ok) + BRANCH(len_ok) → SOLVE(漏洞分支)
```

In [3]:
from z3 import *
addr = BitVec("addr", 8)
len = BitVec("len", 8)
s = Solver()
# 从 SVFG 模板加约束
s.add(addr == 0x5A)
s.add(len > 0)
s.add(len <= 4)
# 求解
s.check()  # 输出 sat
print(s.model())  # 输出 [len = 1, addr = 90]

[len = 4, addr = 90]


SVFG 反映了符号输入（MEM_R）如何通过计算（EQ/GT/AND）影响分支条件（BRANCH），最终导致漏洞触发（SOLVE）。通过分析 SVFG，我们可以构建约束并使用 SMT 求解器找到满足漏洞条件的输入。

固件源码（编译前）

``` python
def run_device():
    addr = sym_read(0)  # 符号输入1
    length = sym_read(1) # 符号输入2

    addr_ok = (addr == 0x5A)  # 分支1
    if not addr_ok: return False

    len_ok = (0 < length <=4) # 分支2
    if not len_ok: return False

    print("BUG触发")
    return True
```

只在需要传值的地方插桩：
``` python
def run_device():
    addr = sym_read(0)
    co3_stub(MEM_R, 0, addr)  # 插桩：传addr的值

    length = sym_read(1)
    co3_stub(MEM_R, 1, length) # 插桩：传length的值

    addr_ok = (addr == 0x5A)
    co3_stub(BRANCH, 1, addr_ok) # 插桩：传分支结果
    if not addr_ok: return False

    len_ok = (0 < length <=4)
    co3_stub(BRANCH, 2, len_ok) # 插桩：传分支结果
    if not len_ok: return False

    print("BUG触发")
    return True
```

MCU 只需要跑这段代码，不需要知道任何 SVFG 的信息。主机编译期生成值地址映射表（SVFG），运行时只负责传值和求解。  

``` Json
# 编译时生成的 SVFG（主机永久保存）
SVFG = {
    # 叶子节点：需要 MCU 传值的地方
    "leaves": [
        {"type": "MEM_R", "offset": 0, "name": "addr"},
        {"type": "MEM_R", "offset": 1, "name": "length"}
    ],
    # 计算节点：编译时已知的运算结构
    "computations": [
        {"id": 1, "op": "EQ", "left": "addr", "right": 0x5A},
        {"id": 2, "op": "GT", "left": "length", "right": 0},
        {"id": 3, "op": "LE", "left": "length", "right": 4}
    ],
    # 根节点：需要求解的分支
    "roots": [
        {"id": 1, "condition": 1},  # 分支1：addr == 0x5A
        {"id": 2, "condition": "AND(2,3)"} # 分支2：length>0 AND length<=4
    ]
}
```

主机做的事（极高效）

1. 收到 MCU 传的 `addr=0x10`、`length=0x09`
2. 从 SVFG 里取出约束模板：
    
    ``` plaintext
    addr == 0x5A
    length > 0 AND length <=4
    ```
    
3. 代入 Z3 求解 → 得到 `addr=0x5A`、`length=0x01`
4. 把新输入发给 MCU，触发漏洞

SVFG 是一个「编号→变量→约束」的映射表，实现了：   
SVFG = 设备端插桩编号 ↔ 主机端符号变量 的双向映射表  

编译固件时，CO3 编译器会给**每一个需要传值的地方**分配一个**唯一的整数编号**：

- 读第 0 个符号输入字节 → 编号 `0`
- 读第 1 个符号输入字节 → 编号 `1`
- 地址校验分支 → 编号 `100`
- 长度校验分支 → 编号 `101`

设备运行时，只需要传**编号 + 值**，不需要传任何变量名或结构信息：

``` plaintext
# 设备发给主机的原始数据（只有数字，极小）
1 0 16    # MEM_R 编号0 值16（0x10）
1 1 9     # MEM_R 编号1 值9（0x09）
3 100 0   # BRANCH 编号100 值0（失败）
```

### 主机端（SVFG 映射）

主机手里的 SVFG 提前存好了**所有编号对应的含义**：  

``` python
# SVFG 核心映射表（编译时生成）
SVFG_ID_MAP = {
    0: "pkt_addr",    # 编号0 → 协议地址变量
    1: "pkt_len",     # 编号1 → 协议长度变量
    100: "branch_addr_ok", # 编号100 → 地址校验分支
    101: "branch_len_ok"   # 编号101 → 长度校验分支
}
```

主机收到设备的数字流后，通过 SVFG 瞬间就能翻译成有意义的变量：

``` plaintext
1 0 16 → pkt_addr = 0x10
1 1 9  → pkt_len = 0x09
3 100 0 → branch_addr_ok = False
```

### 完整的 SVFG 结构（编译时生成）

SVFG 不仅知道「编号对应哪个变量」，还知道**这些变量之间的关系**，也就是**约束公式的结构**。从而能翻转符号：

``` Json
SVFG = {
    # 第一层：编号→变量映射（你的理解）
    "id_map": {
        0: "pkt_addr",
        1: "pkt_len",
        100: "branch_addr_ok",
        101: "branch_len_ok"
    },
    
    # 第二层：变量类型和位数
    "var_types": {
        "pkt_addr": ("BitVec", 8),
        "pkt_len": ("BitVec", 8)
    },
    
    # 第三层：约束公式结构（最核心）
    "constraints": {
        100: "pkt_addr == 0x5A",          # 编号100的分支条件
        101: "pkt_len > 0 AND pkt_len <=4" # 编号101的分支条件
    }
}
```

### 主机端的完整工作流程

1. 收到设备数据：`3 100 0`
2. 通过 `id_map` 翻译：`branch_addr_ok = False`
3. 通过 `constraints` 找到对应的约束：`pkt_addr == 0x5A`
4. 生成 Z3 约束：`s.add(pkt_addr == 0x5A)`
5. 求解得到新输入：`pkt_addr = 0x5A`

**因此主机不需要硬编码任何协议逻辑，只靠 SVFG 就能自动生成约束**。